# SeoulMate RAG + Weather MCP 디버그 노트북

최종 GPT 답변 이전의 전체 과정을 확인합니다.

1. Structured Query와 Restaurant Search Plan
2. RAG에 들어가는 실제 검색 문장과 필터
3. 원본 RAG 후보 최대 30개의 전체 순위와 점수 구성
4. 각 후보의 리뷰·메뉴 근거
5. Weather MCP 원본 응답
6. 날씨 적용 후 전체 순위, 순위 변화, 가감점 이유
7. 선택 후보 상세 근거

`QUESTION_CASE` 셀만 바꾸면 다른 질문도 같은 방식으로 검사할 수 있습니다.

In [1]:
# 1. 절대 경로 기준 실행 환경
import os
import sys
from copy import deepcopy
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

import html
import json
from IPython.display import HTML, display

BACKEND_DIR = Path(r"C:\Users\user\Desktop\seoulmate\SeoulMate\backend")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)

def show_table(rows):
    """pandas 없이 list[dict]를 가로 스크롤 가능한 HTML 표로 표시합니다."""
    rows = list(rows)
    if not rows:
        display(HTML("<em>표시할 행이 없습니다.</em>"))
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    def text(value):
        if isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False, default=str)
        return html.escape(str(value))
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{text(row.get(column, ''))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto;max-width:100%'>"
        "<table style='border-collapse:collapse;font-size:12px'>"
        "<style>th,td{border:1px solid #bbb;padding:5px;vertical-align:top;white-space:nowrap}"
        "td{max-width:420px;white-space:normal}</style>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))

print("backend:", BACKEND_DIR)
print("python:", sys.executable)

backend: C:\Users\user\Desktop\seoulmate\SeoulMate\backend
python: C:\Users\user\anaconda3\python.exe


## 2. 테스트 질문과 고정 Structured Query

실서비스에서는 상위 GPT가 이 JSON을 만듭니다. 여기서는 GPT 파싱 결과의 변동과 비용을 분리하기 위해 JSON을 직접 넣습니다.

In [24]:
# 이 셀을 수정해서 여러 질문을 테스트하세요.
QUESTION_CASE = {
    "language": "ko",
    "intent": "single_place_recommendation",
    "original_question": "맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘",
    "normalized_question": "맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘",
    "tasks": [{
        "task_id": "task_1",
        "domain": "restaurant",
        "search_query": "맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘",
        "themes": ["조용한", "분위기 좋은", "저녁"],
        "desired_count": 3,
        "notes": None,
    }],
    "filters": {
        "location": "종로",
        "radius_km": None,
        "is_active": True,
        "start_date": "2026-07-15",
        "end_date": "2026-07-15",
        "time_window": "evening",
        "party_size": None,
        "budget_min_krw": None,
        "budget_max_krw": None,
        "transportation": [],
        "accessibility": [],
        "required_features": [],
        "excluded_features": [],
    },
    "weather_request": {
        "query": "내일 종로 날씨",
        "location_name": "종로",
        "target_date": "2026-07-15",
        "target_time": "evening",
        "language": "ko",
    },
    "route_context": None,
    "general_response_instruction": None,
}

# 사용자 현재 위치가 서울시청이어도 filters.location=홍대를 우선해야 하는 상황
CURRENT_LAT = 37.5665
CURRENT_LNG = 126.9780
CURRENT_LOCATION_NAME = None
TOP_N = 30

pprint(QUESTION_CASE)

{'filters': {'accessibility': [],
             'budget_max_krw': None,
             'budget_min_krw': None,
             'end_date': '2026-07-15',
             'excluded_features': [],
             'is_active': True,
             'location': '종로',
             'party_size': None,
             'radius_km': None,
             'required_features': [],
             'start_date': '2026-07-15',
             'time_window': 'evening',
             'transportation': []},
 'general_response_instruction': None,
 'intent': 'single_place_recommendation',
 'language': 'ko',
 'normalized_question': '맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘',
 'original_question': '맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘',
 'route_context': None,
 'tasks': [{'desired_count': 3,
            'domain': 'restaurant',
            'notes': None,
            'search_query': '맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘',
            'task_id': 'task_1',
            'themes': ['조용한', '분위기 좋은', '저녁']}],
 'weathe

In [25]:
# 3. Structured Query -> 실제 Restaurant Search Plan
from schemas.structured_query import StructuredTravelQuery
from services.query_policy import derive_source_mode
from services.rag import build_restaurant_search_plan

parsed_query = StructuredTravelQuery.model_validate(QUESTION_CASE)
restaurant_task = next(task for task in parsed_query.tasks if task.domain == "restaurant")
source_mode = derive_source_mode(parsed_query)
plan = build_restaurant_search_plan(
    parsed_query,
    restaurant_task,
    current_lat=CURRENT_LAT,
    current_lng=CURRENT_LNG,
    current_location_name=CURRENT_LOCATION_NAME,
    top_n=TOP_N,
)

print("source_mode:", source_mode)
show_table({"field": key, "value": value} for key, value in asdict(plan).items())

source_mode: rag_mcp


field,value
task_id,task_1
retrieval_query,"맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘 조용한 분위기 좋은 저녁"
review_query,"맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 을 추천해줘 조용한 분위기 좋은 저녁"
menu_query,"맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘"
required_menu_terms,[]
requested_category,일본 요리
location_name,경복궁
origin_lat,37.577613288258206
origin_lng,126.97689786832184
radius_km,2.0


### RAG가 실제 사용하는 값

- `retrieval_query`: 식당·메뉴 임베딩 검색 문장
- `review_query`: 음식 종류를 제거한 리뷰 임베딩 검색 문장
- `requested_category`: 명시적 음식 종류 하드 조건
- `origin_lat/lng`, `radius_km`: DB 후보 공간 범위
- `target_visit_at`: 영업시간 판정 시각
- `required_feature_fields`, `budget_*`: 후보 생성 단계 하드 필터

In [26]:
# 4. RAG 실행 전 입력 요약
rag_input = {
    "task_id": plan.task_id,
    "language_table": "en" if parsed_query.language.lower().startswith("en") else "ko",
    "retrieval_query": plan.retrieval_query,
    "review_query": plan.review_query,
    "requested_category": plan.requested_category,
    "resolved_location": plan.location_name,
    "origin_lat": plan.origin_lat,
    "origin_lng": plan.origin_lng,
    "radius_km": plan.radius_km,
    "open_now": plan.open_now,
    "target_visit_at": plan.target_visit_at,
    "min_rating": plan.min_rating,
    "budget_min_krw": plan.budget_min_krw,
    "budget_max_krw": plan.budget_max_krw,
    "required_feature_fields": plan.required_feature_fields,
    "excluded_feature_fields": plan.excluded_feature_fields,
    "include_weather_features": plan.include_weather_features,
    "top_n": plan.top_n,
}
show_table([rag_input])

task_id,language_table,retrieval_query,review_query,requested_category,resolved_location,origin_lat,origin_lng,radius_km,open_now,target_visit_at,min_rating,budget_min_krw,budget_max_krw,required_feature_fields,excluded_feature_fields,include_weather_features,top_n
task_1,ko,"맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 일식당을 추천해줘 조용한 분위기 좋은 저녁","맵지않고, 적당히 달고 적당히 짠 육류 음식과 야채가 골고루 있는 을 추천해줘 조용한 분위기 좋은 저녁",일본 요리,경복궁,37.577613288258206,126.97689786832184,2.0,False,None,None,None,None,[],[],True,30


In [27]:
# 5. 원본 RAG 검색 실행 (MCP 재랭킹 전)
from services.rag import search_restaurants_structured

rag_result = search_restaurants_structured(
    parsed_query,
    restaurant_task,
    current_lat=CURRENT_LAT,
    current_lng=CURRENT_LNG,
    current_location_name=CURRENT_LOCATION_NAME,
    top_n=TOP_N,
)
raw_candidates = deepcopy(rag_result["candidates"])

print("검색 위치:", rag_result.get("location_name"), rag_result.get("origin_lat"), rag_result.get("origin_lng"))
print("추출 음식 종류:", rag_result.get("extracted_category"))
print("RAG 후보 수:", len(raw_candidates))

검색 위치: 경복궁 37.577613288258206 126.97689786832184
추출 음식 종류: 일본 요리
RAG 후보 수: 9


In [28]:
# 6. MCP 적용 전 전체 RAG 순위와 점수 구성
#    breakdown 키는 원본판(restaurant_rrf/review_rrf/menu_rrf/category_adjustment/category_status)과
#    패치판(base_score/category_boost/rating_preference_boost/category_confidence)이 다르다.
#    아래 헬퍼가 양쪽을 모두 안전하게 읽는다.
def bget(b, key, default=0.0):
    v = b.get(key)
    return default if v is None else v

def base_score_of(b):
    # 패치판은 base_score 를 저장. 원본판은 3종 RRF 합이 base 다.
    if b.get("base_score") is not None:
        return b["base_score"]
    return bget(b, "restaurant_rrf") + bget(b, "review_rrf") + bget(b, "menu_rrf")

def category_boost_of(b):
    # 패치판 category_boost, 원본판 category_adjustment (이름만 다름)
    v = b.get("category_boost")
    if v is None:
        v = b.get("category_adjustment")
    return 0.0 if v is None else v

def compact_reviews(candidate):
    return " | ".join(
        f"{r.get('similarity', 0):.3f}:{str(r.get('content', ''))[:80]}"
        for r in candidate.get("evidence", {}).get("reviews", [])
    )

def compact_menus(candidate):
    return " | ".join(
        f"{m.get('similarity', 0):.3f}:{m.get('menu_name', '')}"
        for m in candidate.get("evidence", {}).get("menus", [])
    )

def rag_rows(candidates):
    rows = []
    for rank, c in enumerate(candidates, 1):
        b = c.get("breakdown", {})
        rows.append({
            "rag_rank": rank,
            "restaurant_id": c["restaurant_id"],
            "name": c["name"],
            "category": c.get("category"),
            "category_kakao": c.get("category_kakao"),
            "category_status": b.get("category_status"),
            "category_confidence": b.get("category_confidence"),   # 원본판이면 None
            "rag_score": c.get("score"),
            "restaurant_rrf": bget(b, "restaurant_rrf"),
            "review_rrf": bget(b, "review_rrf"),
            "menu_rrf": bget(b, "menu_rrf"),
            "base_score": base_score_of(b),
            "category_boost": category_boost_of(b),
            "rating_preference_boost": bget(b, "rating_preference_boost"),
            "distance_km": c.get("distance_km"),
            "rating": c.get("rating"),
            "review_count": c.get("review_count"),
            "open_at_requested_time": c.get("open_status"),
            "open_status_basis": c.get("open_status_basis"),
            "menu_price_median": c.get("menu_price_median"),
            "parking": c.get("has_parking"),
            "pets": c.get("allows_pets"),
            "group_seating": c.get("has_group_seating"),
            "private_room": c.get("has_private_room"),
            "top_review_evidence": compact_reviews(c),
            "top_menu_evidence": compact_menus(c),
        })
    return rows

rag_rows_data = rag_rows(raw_candidates)
show_table(rag_rows_data)

rag_rank,restaurant_id,name,category,category_kakao,category_status,category_confidence,rag_score,restaurant_rrf,review_rrf,menu_rrf,base_score,category_boost,rating_preference_boost,distance_km,rating,review_count,open_at_requested_time,open_status_basis,menu_price_median,parking,pets,group_seating,private_room,top_review_evidence,top_menu_evidence
1,5035631,타니 넥스트도어,"일본 요리, 스시, 아시아 요리",None,match,inferred,0.06942397503191941,0.0,0.06428145836288834,0.0,0.06428145836288834,0.005142516669031067,0.0,1.5396044164766696,4.6,23,True,None,None,None,None,None,None,"0.568:정교하게 준비된 음식, 아늑한 분위기와 친절한 직원들의 완벽한 조화 | 0.543:좋은 서비스, 편안한 분위기 맛있는 음식! | 0.537:좋은 느낌. 잘 먹었습니다. 모든 것이 굉장했다 음식, 분위기 및 서비스 관점에서 적극 권장됩니다.",
2,2422441,스시 조,"일본 요리, 스시, 아시아 요리","초밥,롤",match,confirmed,0.02672819566220581,0.0,0.02227349638517151,0.0,0.02227349638517151,0.004454699277034302,0.0,1.4841399207724986,3.9,74,True,None,20500,True,None,None,True,"0.516:친구들과 함께 즐겁게 저녁 식사. 우리 둘은 메밀 껍데기 를 주문하면 - 하나 하나는 뜨거운 날씨. 제가 사랑하는 내 골드 준비. 친구 주문한 | 0.511:맛있고, 친절하고, 분위기 좋다. 고급이다. 가격은 세지만 최소한 2시간 이상 동안 훌륭한 맛을 느끼고, 극진한 대접을 받으며, 분위기에 취할수",
3,4030880,오가와,"일본 요리, 스시, 아시아 요리",None,match,inferred,0.018123317019497254,0.004635761589403973,0.006825938566552901,0.005319148936170213,0.016780849092127088,0.001342467927370167,0.0,0.6848998160170054,4.7,32,None,None,85000,False,None,True,None,"0.497:지하 푸드 코트에 도착하면 작은 나무 문 뒤에 레스토랑이 숨겨져 있습니다. 우리는 2 저녁 서비스를 위해 오후 7시 40 분에 도착했고, 우리",0.453:초밥도시락
4,5021758,라따블,"이탈리아 요리, 일본 요리, 유럽 요리, 아시아 요리, 한국",None,match,inferred,0.009166763173132598,0.0053030303030303025,0.0,0.0031847133757961785,0.00848774367882648,0.0006790194943061184,0.0,1.5968853719375529,4.3,36,False,None,37400,True,None,None,None,,0.427:조식 뷔페
5,34275254,토리노라스베가스,"일본 요리, 아시아 요리, 한국, 일본식 퓨전",일본식주점,match,confirmed,0.007792207792207793,0.0,0.006493506493506494,0.0,0.006493506493506494,0.001298701298701299,0.0,1.7858416270409938,5.0,15,True,None,19500,None,None,None,None,0.496:을자로에서 저녁 술자리를 찾다가 토리노라스베가스에 방문했습니다. 매장은 일본 골목 술집처럼 꾸며져 있어 여행 중 들른 사람에게도 분위기가 잘 전,
6,16712513,아키라백 서울,"일본 요리, 스시, 아시아 요리",일식집,match,confirmed,0.007317073170731708,0.0,0.0,0.006097560975609756,0.006097560975609756,0.0012195121951219514,0.0,0.8113930953717273,4.2,22,True,None,129000,None,None,None,None,,0.461:Chef Shin's Recommendations
7,12979587,애슐리퀸즈 - 종각역점,"중국 요리, 일본 요리, 미국 요리, 아시아 요리, 한국",패밀리레스토랑,match,inferred,0.006870878437227821,0.0,0.0,0.006361924478914649,0.006361924478914649,0.0005089539583131719,0.0,0.9944757169568261,4.7,6,True,None,15900,True,None,None,None,,0.420:평일 디너 | 0.410:평일 런치 | 0.400:평일 디너 초등학생
8,31720251,Jongno 24 Hours Ramen Convenience Store,일본 요리,None,match,inferred,0.00525,0.004861111111111111,0.0,0.0,0.004861111111111111,0.0003888888888888889,0.0,1.588247178212764,3.7,3,True,None,None,None,None,None,None,,
9,20260781,모도우 광화문점,"해산물, 아시아 요리, 한국, 일본식 퓨전",샤브샤브,match,inferred,0.004615384615384616,0.0,0.0,0.004273504273504274,0.004273504273504274,0.00034188034188034193,0.0,0.17065423692004983,5.0,14,True,None,55000,True,None,None,None,,0.439:농어구이와 보리리조또


### RAG 점수 구성 분해 (식당+리뷰+메뉴 벡터 → base → 부스팅 → 최종)

각 후보의 최종 `rag_score` 가 어떻게 만들어졌는지 단계별로 본다.

- **base = 식당RRF + 리뷰RRF + 메뉴RRF** (세 벡터 검색의 RRF 가중합)
- **최종 = base + 카테고리 부스팅 + 평점선호 부스팅**
- `합계검산` 열이 최종과 일치(✅)하면 점수 구성이 설명과 맞는다는 뜻이다.
- 원본판 rag.py 는 `category_confidence`/`rating_preference_boost` 를 저장하지 않으므로 그 열은 0/None 으로 보일 수 있다(정상).

In [29]:
# 6-1. 점수 구성 분해 + 검산
def pct(part, whole):
    return f"{(part / whole * 100):.1f}%" if whole else "-"

score_breakdown_rows = []
for rank, c in enumerate(raw_candidates, 1):
    b = c.get("breakdown", {})
    rest = bget(b, "restaurant_rrf")
    rev = bget(b, "review_rrf")
    menu = bget(b, "menu_rrf")
    base = base_score_of(b)
    cat_boost = category_boost_of(b)
    rate_boost = bget(b, "rating_preference_boost")
    final = c.get("score", base + cat_boost + rate_boost)
    recomputed = base + cat_boost + rate_boost
    match = abs(recomputed - final) < 1e-9
    score_breakdown_rows.append({
        "순위": rank,
        "이름": c["name"],
        "① 식당RRF": round(rest, 6),
        "② 리뷰RRF": round(rev, 6),
        "③ 메뉴RRF": round(menu, 6),
        "식당%": pct(rest, base),
        "리뷰%": pct(rev, base),
        "메뉴%": pct(menu, base),
        "base(①+②+③)": round(base, 6),
        "카테status": b.get("category_status"),
        "카테부스팅": round(cat_boost, 6),
        "평점부스팅": round(rate_boost, 6),
        "최종 rag_score": round(final, 6),
        "합계검산": "✅" if match else f"❌ {round(recomputed,6)}",
    })

show_table(score_breakdown_rows)

# 부스팅이 실제 순위를 바꾸는지: base 만으로 정렬한 순서와 비교
with_boost = [c["name"] for c in raw_candidates]
without_boost = [
    c["name"] for c in sorted(
        raw_candidates,
        key=lambda c: (-base_score_of(c.get("breakdown", {})),
                       -(c.get("review_count") or 0), c["restaurant_id"]),
    )
]
print("부스팅 적용 순위 :", with_boost[:8])
print("base만 정렬 순위 :", without_boost[:8])
print("부스팅이 순위를 바꿨는가:", with_boost != without_boost)
if with_boost == without_boost:
    print("  -> 부스팅이 순위에 영향 없음. 최종 후보가 모두 같은 category_status 라 "
          "균일 배율이 되어 상쇄된 경우다(원본 rag.py 의 알려진 특성).")

순위,이름,① 식당RRF,② 리뷰RRF,③ 메뉴RRF,식당%,리뷰%,메뉴%,base(①+②+③),카테status,카테부스팅,평점부스팅,최종 rag_score,합계검산
1,타니 넥스트도어,0.0,0.064281,0.0,0.0%,100.0%,0.0%,0.064281,match,0.005143,0.0,0.069424,✅
2,스시 조,0.0,0.022273,0.0,0.0%,100.0%,0.0%,0.022273,match,0.004455,0.0,0.026728,✅
3,오가와,0.004636,0.006826,0.005319,27.6%,40.7%,31.7%,0.016781,match,0.001342,0.0,0.018123,✅
4,라따블,0.005303,0.0,0.003185,62.5%,0.0%,37.5%,0.008488,match,0.000679,0.0,0.009167,✅
5,토리노라스베가스,0.0,0.006494,0.0,0.0%,100.0%,0.0%,0.006494,match,0.001299,0.0,0.007792,✅
6,아키라백 서울,0.0,0.0,0.006098,0.0%,0.0%,100.0%,0.006098,match,0.00122,0.0,0.007317,✅
7,애슐리퀸즈 - 종각역점,0.0,0.0,0.006362,0.0%,0.0%,100.0%,0.006362,match,0.000509,0.0,0.006871,✅
8,Jongno 24 Hours Ramen Convenience Store,0.004861,0.0,0.0,100.0%,0.0%,0.0%,0.004861,match,0.000389,0.0,0.00525,✅
9,모도우 광화문점,0.0,0.0,0.004274,0.0%,0.0%,100.0%,0.004274,match,0.000342,0.0,0.004615,✅


부스팅 적용 순위 : ['타니 넥스트도어', '스시 조', '오가와', '라따블', '토리노라스베가스', '아키라백 서울', '애슐리퀸즈 - 종각역점', 'Jongno 24 Hours Ramen Convenience Store']
base만 정렬 순위 : ['타니 넥스트도어', '스시 조', '오가와', '라따블', '토리노라스베가스', '애슐리퀸즈 - 종각역점', '아키라백 서울', 'Jongno 24 Hours Ramen Convenience Store']
부스팅이 순위를 바꿨는가: True


### 필터 작동 확인 (plan 조건이 실제 후보에 적용됐나)

`plan` 에 들어간 하드필터(반경/평점/시설/예산/카테고리)가 실제 후보에서 지켜졌는지 수치로 대조한다.
필터는 DB 쿼리 단계에서 걸러지므로, 위반 건수가 0 이어야 정상이다.

In [30]:
# 6-2. 필터 작동 확인
filter_rows = []
cands = raw_candidates
n = len(cands)

def frac(pred):
    return f"{sum(1 for c in cands if pred(c))}/{n}" if n else "0/0"

# 반경
if plan.origin_lat is not None and plan.radius_km:
    viol = [c for c in cands if c.get("distance_km") is not None and c["distance_km"] > plan.radius_km + 0.01]
    filter_rows.append({"필터": f"반경 <= {plan.radius_km}km", "위반건수": len(viol),
                        "충족": frac(lambda c: c.get("distance_km") is None or c["distance_km"] <= plan.radius_km + 0.01),
                        "후보내최대거리": round(max((c.get("distance_km") or 0) for c in cands), 2) if n else None})
# 평점 하한
if plan.min_rating is not None:
    viol = [c for c in cands if not (c.get("rating") is not None and c["rating"] >= plan.min_rating)]
    filter_rows.append({"필터": f"평점 >= {plan.min_rating}", "위반건수": len(viol),
                        "충족": frac(lambda c: (c.get("rating") or 0) >= plan.min_rating),
                        "후보내최저평점": min((c.get("rating") or 99) for c in cands) if n else None})
# 필수 시설
for field in plan.required_feature_fields:
    viol = [c for c in cands if c.get(field) is not True]
    filter_rows.append({"필터": f"{field} = True(필수)", "위반건수": len(viol),
                        "충족": frac(lambda c, f=field: c.get(f) is True), "후보내최대거리": None})
# 제외 시설
for field in plan.excluded_feature_fields:
    viol = [c for c in cands if c.get(field) is True]
    filter_rows.append({"필터": f"{field} = True 제외", "위반건수": len(viol),
                        "충족": frac(lambda c, f=field: c.get(f) is not True), "후보내최대거리": None})
# 예산 상한
if plan.budget_max_krw is not None:
    viol = [c for c in cands if not (c.get("menu_price_median") is not None and c["menu_price_median"] <= plan.budget_max_krw)]
    filter_rows.append({"필터": f"중앙가 <= {plan.budget_max_krw}", "위반건수": len(viol),
                        "충족": frac(lambda c: (c.get("menu_price_median") or 1e9) <= plan.budget_max_krw), "후보내최대거리": None})
# 예산 하한
if plan.budget_min_krw is not None:
    viol = [c for c in cands if not (c.get("menu_price_median") is not None and c["menu_price_median"] >= plan.budget_min_krw)]
    filter_rows.append({"필터": f"중앙가 >= {plan.budget_min_krw}", "위반건수": len(viol),
                        "충족": frac(lambda c: (c.get("menu_price_median") or 0) >= plan.budget_min_krw), "후보내최대거리": None})
# 카테고리 mismatch 제외
if plan.requested_category:
    viol = [c for c in cands if c.get("breakdown", {}).get("category_status") == "mismatch"]
    filter_rows.append({"필터": f"카테고리 '{plan.requested_category}' mismatch 제외", "위반건수": len(viol),
                        "충족": frac(lambda c: c.get("breakdown", {}).get("category_status") != "mismatch"), "후보내최대거리": None})
# 영업 시간(요청 시각/현재)
if plan.open_now or plan.target_visit_at:
    basis = "요청시각" if plan.target_visit_at else "현재"
    viol = [c for c in cands if c.get("open_status") is False]
    filter_rows.append({"필터": f"영업중({basis})", "위반건수": len(viol),
                        "충족": frac(lambda c: c.get("open_status") is not False), "후보내최대거리": None})

if filter_rows:
    show_table(filter_rows)
else:
    print("적용된 하드필터가 없습니다 (지역/카테고리만으로 검색).")

필터,위반건수,충족,후보내최대거리
반경 <= 2.0km,0,9/9,1.79
카테고리 '일본 요리' mismatch 제외,0,9/9,None


In [32]:
# 7. 후보별 리뷰/메뉴 근거를 행 단위로 펼쳐서 확인
evidence_rows = []
for rank, candidate in enumerate(raw_candidates, 1):
    for review in candidate.get("evidence", {}).get("reviews", []):
        evidence_rows.append({
            "rag_rank": rank,
            "restaurant_id": candidate["restaurant_id"],
            "name": candidate["name"],
            "evidence_type": "review",
            "global_vector_rank": review.get("rank"),
            "similarity": review.get("similarity"),
            "is_main": None,
            "content": review.get("content"),
        })
    for menu in candidate.get("evidence", {}).get("menus", []):
        evidence_rows.append({
            "rag_rank": rank,
            "restaurant_id": candidate["restaurant_id"],
            "name": candidate["name"],
            "evidence_type": "menu",
            "global_vector_rank": menu.get("rank"),
            "similarity": menu.get("similarity"),
            "is_main": menu.get("is_main"),
            "content": menu.get("menu_name"),
        })

show_table(evidence_rows)

rag_rank,restaurant_id,name,evidence_type,global_vector_rank,similarity,is_main,content
1,5035631,타니 넥스트도어,review,10,0.5684940696951175,None,"정교하게 준비된 음식, 아늑한 분위기와 친절한 직원들의 완벽한 조화"
1,5035631,타니 넥스트도어,review,35,0.5433990945210496,None,"좋은 서비스, 편안한 분위기 맛있는 음식!"
1,5035631,타니 넥스트도어,review,46,0.5369996197155504,None,"좋은 느낌. 잘 먹었습니다. 모든 것이 굉장했다 음식, 분위기 및 서비스 관점에서 적극 권장됩니다."
2,2422441,스시 조,review,105,0.5164114872610017,None,친구들과 함께 즐겁게 저녁 식사. 우리 둘은 메밀 껍데기 를 주문하면 - 하나 하나는 뜨거운 날씨. 제가 사랑하는 내 골드 준비. 친구 주문한 우동 두 개의 객실 안전. 훌륭하고 완벽한 준비. 서비스는 아주 좋았습니다.
2,2422441,스시 조,review,137,0.5112091244177126,None,"맛있고, 친절하고, 분위기 좋다. 고급이다. 가격은 세지만 최소한 2시간 이상 동안 훌륭한 맛을 느끼고, 극진한 대접을 받으며, 분위기에 취할수 있다. 고급스러운 플레이팅과 내부 인테리어, 그릇에 시각적인 즐거움도 한몫한다."
3,4030880,오가와,review,233,0.4967697782737641,None,"지하 푸드 코트에 도착하면 작은 나무 문 뒤에 레스토랑이 숨겨져 있습니다. 우리는 2 저녁 서비스를 위해 오후 7시 40 분에 도착했고, 우리 앞에 이미 6 명이 기다리고 있었으므로 그렇게 할 권리가있었습니다. 오후 8시에 들어가면 주방장 테이블에 20 칸의 공간이 있습니다. 3 요리사가 당신의 미쳤고 맛있는 초밥 발견 여행을 돌보고, 무엇을 먹고 있는지 알려주고, 일이 올바르게 수행되는 방법을 보여줍니다 (우리가 아시아 요리의 초보자라면). 음료는 꽤 비싸지 만 (맥주는 12k 원) 차와 된장국은 무제한입니다. 나는 남자 친구를 생일에 데려갔습니다. 그는 초밥을 좋아했습니다. 그리고 우리가 식당에 들어갔을 때 그의 얼굴은 모든 가치가있었습니다. 우리가 가졌던 최고의 스시 저녁 식사와 두 가지 의견 모두에서 우리는 결코 잊지 못할 생일 저녁입니다. 정보 : 1 . 점심 메뉴 요금은 1 인당 40 k 원, 서비스 : 12-1 pm 또는 1 : 10-2 : 10 pm 2. 석식 메뉴 요금은 1 인당 70k 원, 서비스 : 오후 6시-8시, 오후 8시-10시 미리 거기에 -좌석 확보, 레스토랑은 목요일부터 일요일까지 꽉 참 -점심 식사 시간은 1 시간, 저녁 식사 여행은 2 시간입니다! ! 추신 : 음식 애호가로서 유럽의 1 미쉐린 스타 레스토랑보다 정말 잘 먹었습니다."
3,4030880,오가와,menu,34,0.45291373829736825,True,초밥도시락
4,5021758,라따블,menu,97,0.42699721842170413,True,조식 뷔페
5,34275254,토리노라스베가스,review,248,0.4956336264847818,None,"을자로에서 저녁 술자리를 찾다가 토리노라스베가스에 방문했습니다. 매장은 일본 골목 술집처럼 꾸며져 있어 여행 중 들른 사람에게도 분위기가 잘 전달될 것 같았습니다. 야키토리는 숯 향이 자연스럽게 배어 맥주와 잘 어울렸고, 쿠시카츠는 바삭하게 먹기 좋아 여러 개를 나눠 주문하기 편했습니다. 참치회가 있어 튀김이나 꼬치만 먹을 때보다 테이블이 더 풍성해졌습니다. 을지로에서 캐주얼한 데이트나 친구 모임 장소를 찾는다면 추천할 만합니다. 직원 안내가 부담스럽지 않아 외국인 손님도 메뉴를 고르기 어렵지 않아 보였습니다. 골목 분위기까지 함께 즐길 수 있어 다음 서울 일정에도 다시 들르고 싶습니다. 술을 많이 마시지 않아도 안주만으로 만족스러운 저녁이었습니다."
6,16712513,아키라백 서울,menu,22,0.46090595136806023,True,Chef Shin's Recommendations


In [33]:
# 8. Weather MCP 원본 응답
from services.weather_mcp_client import get_weather_via_mcp

weather_query = (
    parsed_query.weather_request.query
    if parsed_query.weather_request
    else parsed_query.original_question
)
weather = await get_weather_via_mcp(
    weather_query,
    plan.origin_lat,
    plan.origin_lng,
    parsed_query.language,
    plan.location_name,
)
pprint(weather)

if not weather.get("available"):
    print("주의: MCP 서버 또는 기상청 API가 응답하지 않아 순위 변화가 없을 수 있습니다.")

{'available': True,
 'base_at': '2026-07-15T17:00:00+09:00',
 'condition': 'clear',
 'condition_label': '강수 없음',
 'feels_like': 'hot',
 'feels_like_label': '더움',
 'forecast_for': '2026-07-16T19:00:00+09:00',
 'forecast_offset_minutes': 0,
 'humidity_pct': 65.0,
 'is_forecast': True,
 'language': 'ko',
 'location': {'lat': 37.577613288258206,
              'lng': 126.97689786832184,
              'nx': 60,
              'ny': 127},
 'place_name': '경복궁',
 'precipitation_probability_pct': 0.0,
 'rainfall_mm': 0.0,
 'recommendation_policy': '날씨는 보조 근거로만 사용하고, 사용자 필수 조건을 무시하거나 후보 데이터에 없는 시설을 '
                          '추측하지 마세요.',
 'requested_for': '2026-07-16T19:00:00+09:00',
 'resolved_query': '내일 종로 날씨',
 'should_affect_recommendation': True,
 'sky': 'clear',
 'sky_label': '맑음',
 'source': 'KMA-vilage-forecast',
 'summary': '내일 저녁(기본 19시): 강수 없음, 28°C입니다.',
 'target_label': '내일 저녁(기본 19시)',
 'temperature_c': 28.0,
 'usage_guidance': ['냉방이 잘 되는 실내 공간은 보조 장점이 될 수 있습니다.'],
 'weather_tags':

In [34]:
# 9. Weather MCP 적용 후 전체 순위와 변경 이유
from services.weather_reranker import rerank_with_weather

weather_candidates = rerank_with_weather(
    deepcopy(raw_candidates),
    weather,
    parsed_query.original_question,
    source_mode="rag_mcp",
)

raw_rank_by_id = {c["restaurant_id"]: rank for rank, c in enumerate(raw_candidates, 1)}
raw_score_by_id = {c["restaurant_id"]: c["score"] for c in raw_candidates}

comparison_rows = []
for mcp_rank, c in enumerate(weather_candidates, 1):
    restaurant_id = c["restaurant_id"]
    raw_rank = raw_rank_by_id[restaurant_id]
    features = c.get("weather_features") or {}
    comparison_rows.append({
        "mcp_rank": mcp_rank,
        "rag_rank": raw_rank,
        "rank_change": raw_rank - mcp_rank,
        "restaurant_id": restaurant_id,
        "name": c["name"],
        "raw_rag_score": raw_score_by_id[restaurant_id],
        "weather_score": c.get("weather_score"),
        "final_score": c.get("score"),
        "weather_reasons": " | ".join(c.get("weather_reasons", [])),
        "distance_km": c.get("distance_km"),
        "parking": c.get("has_parking"),
        "outdoor_confidence": c.get("outdoor_confidence"),
        "warm_menu": features.get("has_warm_menu"),
        "warm_matches": ", ".join(features.get("warm_menu_matches", [])),
        "cool_menu": features.get("has_cool_menu"),
        "cool_matches": ", ".join(features.get("cool_menu_matches", [])),
    })

show_table(comparison_rows)

print("순위 상승: rank_change > 0")
print("순위 하락: rank_change < 0")

mcp_rank,rag_rank,rank_change,restaurant_id,name,raw_rag_score,weather_score,final_score,weather_reasons,distance_km,parking,outdoor_confidence,warm_menu,warm_matches,cool_menu,cool_matches
1,1,0,5035631,타니 넥스트도어,0.06942397503191941,0.5,0.9500000000000001,,1.5396044164766696,None,unknown,False,,False,
2,2,0,2422441,스시 조,0.02672819566220581,0.5,0.3964995498302304,,1.4841399207724986,True,unknown,False,,False,
3,3,0,4030880,오가와,0.018123317019497254,0.5,0.28494744157257124,,0.6848998160170054,False,unknown,False,,False,
4,4,0,5021758,라따블,0.009166763173132598,0.5,0.16883627885073066,,1.5968853719375529,True,unknown,False,,False,
5,5,0,34275254,토리노라스베가스,0.007792207792207793,0.5,0.15101678864920393,,1.7858416270409938,None,unknown,False,,False,
6,6,0,16712513,아키라백 서울,0.007317073170731708,0.5,0.14485722836571588,,0.8113930953717273,None,unknown,False,,False,
7,7,0,12979587,애슐리퀸즈 - 종각역점,0.006870878437227821,0.5,0.1390728396157363,,0.9944757169568261,True,unknown,False,,False,
8,8,0,31720251,Jongno 24 Hours Ramen Convenience Store,0.00525,0.5,0.11806006135240114,,1.588247178212764,None,unknown,False,,False,
9,9,0,20260781,모도우 광화문점,0.004615384615384616,0.5,0.10983302096914387,,0.17065423692004983,True,unknown,True,"한우샤브샤브코스 (120g), 한우샤브샤브스페셜코스 (90g), 한우샤브샤브코스 (90g)",False,


순위 상승: rank_change > 0
순위 하락: rank_change < 0


In [35]:
# 10. 실제로 순위가 바뀐 후보와 날씨 이유만 보기
changed_rows = [
    row for row in comparison_rows
    if row["rank_change"] != 0 or row["weather_reasons"]
]
show_table(changed_rows)

In [36]:
# 11. 합성 날씨 시나리오 회귀 디버깅
# 실제 예보와 별개로 더움·추움·비·비+강풍 규칙이 과도하게 작동하는지 확인합니다.
SYNTHETIC_WEATHER_SCENARIOS = {
    "hot_30c": {
        "available": True, "condition": "clear", "temperature_c": 30.0, "wind_speed_mps": 2.0,
    },
    "cold_0c": {
        "available": True, "condition": "clear", "temperature_c": 0.0, "wind_speed_mps": 2.0,
    },
    "rain": {
        "available": True, "condition": "rain", "temperature_c": 18.0, "wind_speed_mps": 3.0,
    },
    "rain_strong_wind": {
        "available": True, "condition": "rain", "temperature_c": 18.0, "wind_speed_mps": 9.0,
    },
}

scenario_summary = []
scenario_reason_rows = []
original_ids = [c["restaurant_id"] for c in raw_candidates]
original_rank = {restaurant_id: rank for rank, restaurant_id in enumerate(original_ids, 1)}

for scenario_name, scenario_weather in SYNTHETIC_WEATHER_SCENARIOS.items():
    reranked = rerank_with_weather(
        deepcopy(raw_candidates), scenario_weather,
        parsed_query.original_question, source_mode="rag_mcp",
    )
    changes = [
        original_rank[c["restaurant_id"]] - rank
        for rank, c in enumerate(reranked, 1)
    ]
    scenario_summary.append({
        "scenario": scenario_name,
        "candidates": len(reranked),
        "changed_candidates": sum(change != 0 for change in changes),
        "largest_rise": max(changes, default=0),
        "largest_drop": min(changes, default=0),
        "candidates_with_reasons": sum(bool(c.get("weather_reasons")) for c in reranked),
        "top5": " | ".join(c["name"] for c in reranked[:5]),
    })
    for rank, c in enumerate(reranked, 1):
        if c.get("weather_reasons"):
            scenario_reason_rows.append({
                "scenario": scenario_name,
                "mcp_rank": rank,
                "rag_rank": original_rank[c["restaurant_id"]],
                "rank_change": original_rank[c["restaurant_id"]] - rank,
                "name": c["name"],
                "weather_score": c.get("weather_score"),
                "reasons": " | ".join(c.get("weather_reasons", [])),
            })

show_table(scenario_summary)
show_table(scenario_reason_rows)

scenario,candidates,changed_candidates,largest_rise,largest_drop,candidates_with_reasons,top5
hot_30c,9,0,0,0,0,타니 넥스트도어 | 스시 조 | 오가와 | 라따블 | 토리노라스베가스
cold_0c,9,2,1,-1,1,타니 넥스트도어 | 스시 조 | 오가와 | 라따블 | 토리노라스베가스
rain,9,4,2,-2,9,타니 넥스트도어 | 스시 조 | 오가와 | 라따블 | 애슐리퀸즈 - 종각역점
rain_strong_wind,9,4,2,-2,9,타니 넥스트도어 | 스시 조 | 오가와 | 라따블 | 애슐리퀸즈 - 종각역점


scenario,mcp_rank,rag_rank,rank_change,name,weather_score,reasons
cold_0c,8,9,1,모도우 광화문점,0.65,"추운 날 어울리는 따뜻한 메뉴가 있음 (한우샤브샤브코스 (120g), 한우샤브샤브스페셜코스 (90g), 한우샤브샤브코스 (90g))"
rain,1,1,0,타니 넥스트도어,0.45,비나 눈이 올 때 직선거리 기준 이동 거리가 다소 멂
rain,2,2,0,스시 조,0.6,비나 눈이 올 때 편리한 주차 가능
rain,3,3,0,오가와,0.5700000000000001,비나 눈이 올 때 이동 부담이 적은 1km 이내 거리
rain,4,4,0,라따블,0.55,비나 눈이 올 때 직선거리 기준 이동 거리가 다소 멂 | 비나 눈이 올 때 편리한 주차 가능
rain,5,7,2,애슐리퀸즈 - 종각역점,0.67,비나 눈이 올 때 이동 부담이 적은 1km 이내 거리 | 비나 눈이 올 때 편리한 주차 가능
rain,6,6,0,아키라백 서울,0.5700000000000001,비나 눈이 올 때 이동 부담이 적은 1km 이내 거리
rain,7,5,-2,토리노라스베가스,0.45,비나 눈이 올 때 직선거리 기준 이동 거리가 다소 멂
rain,8,9,1,모도우 광화문점,0.72,비나 눈이 올 때 이동 부담이 매우 적은 500m 이내 거리 | 비나 눈이 올 때 편리한 주차 가능
rain,9,8,-1,Jongno 24 Hours Ramen Convenience Store,0.45,비나 눈이 올 때 직선거리 기준 이동 거리가 다소 멂


In [37]:
# 12. 특정 후보의 모든 원본 값을 상세 확인
# 보고 싶은 순위 또는 restaurant_id로 바꾸세요.
DEBUG_MCP_RANK = 1
DEBUG_RESTAURANT_ID = None

if DEBUG_RESTAURANT_ID is None:
    selected = weather_candidates[DEBUG_MCP_RANK - 1]
else:
    selected = next(c for c in weather_candidates if c["restaurant_id"] == DEBUG_RESTAURANT_ID)

pprint(selected, sort_dicts=False)

{'restaurant_id': 5035631,
 'name': '타니 넥스트도어',
 'category': '일본 요리, 스시, 아시아 요리',
 'category_kakao': None,
 'description': None,
 'description_kakao': None,
 'rating': 4.6,
 'review_count': 23,
 'hours': '월 11:30~22:00 / 화 11:30~22:00 / 수 11:30~22:00 / 목 11:30~22:00 / 금 '
          '11:30~22:00 / 토 11:30~22:00 / 일 11:30~22:00',
 'open_status': True,
 'open_status_basis': None,
 'address': '서울 중구 남대문로 73 롯데백화점에비뉴엘 9층',
 'image': 'https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/f7/80/ce/caption.jpg?w=200&h=-1&s=1&cx=1920&cy=1079&chk=v1_5192524b3b14f2153843',
 'lat': 37.564266,
 'lng': 126.981544,
 'distance_km': 1.5396044164766696,
 'has_parking': None,
 'allows_pets': None,
 'has_kids_menu': None,
 'has_group_seating': None,
 'has_private_room': None,
 'has_baby_chair': None,
 'has_disabled_access': None,
 'menu_price_min': None,
 'menu_price_median': None,
 'score': 0.9500000000000001,
 'breakdown': {'restaurant_rrf': 0.0,
               'review_rrf': 0.06428145836288834,
  

In [15]:
# 13. RAG_ONLY 안전성 확인: 날씨 조회/재랭킹 없이 원래 순서가 유지되어야 함
from services.weather_reranker import prepare_rag_only_candidates

rag_only_candidates = prepare_rag_only_candidates(deepcopy(raw_candidates))
assert [c["restaurant_id"] for c in rag_only_candidates] == [c["restaurant_id"] for c in raw_candidates]
assert all(c.get("weather_score") is None for c in rag_only_candidates)
assert all(not c.get("weather_reasons") for c in rag_only_candidates)
print("RAG_ONLY 순서 보존 및 날씨 값 제거 확인 완료")

RAG_ONLY 순서 보존 및 날씨 값 제거 확인 완료


In [16]:
# 14. 선택 사항: 최종 GPT 답변까지 생성
# RAG/MCP 디버깅만 할 때는 False로 두세요.
RUN_FINAL_GPT = False

if RUN_FINAL_GPT:
    from services.llm import generate_recommendation
    final_answer = generate_recommendation(
        parsed_query.original_question,
        parsed_query.language,
        weather_candidates[:10],
        weather,
        source_mode="rag_mcp",
    )
    print(final_answer)
else:
    print("최종 GPT 호출 생략: RUN_FINAL_GPT=True로 바꾸면 실행됩니다.")

최종 GPT 호출 생략: RUN_FINAL_GPT=True로 바꾸면 실행됩니다.


In [17]:
# 15. 선택 사항: 이번 디버그 결과를 JSON으로 저장
SAVE_TRACE = False
TRACE_PATH = BACKEND_DIR / "debug_outputs" / "rag_mcp_trace.json"

if SAVE_TRACE:
    import json
    TRACE_PATH.parent.mkdir(parents=True, exist_ok=True)
    trace = {
        "question_case": QUESTION_CASE,
        "search_plan": {key: str(value) if key == "target_visit_at" else value for key, value in asdict(plan).items()},
        "rag_result_meta": {key: value for key, value in rag_result.items() if key != "candidates"},
        "rag_candidates": raw_candidates,
        "weather": weather,
        "weather_reranked_candidates": weather_candidates,
    }
    TRACE_PATH.write_text(json.dumps(trace, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print("saved:", TRACE_PATH)
else:
    print("저장하지 않음: SAVE_TRACE=True로 바꾸면 JSON trace가 생성됩니다.")

저장하지 않음: SAVE_TRACE=True로 바꾸면 JSON trace가 생성됩니다.
